# LLM Agents 

# System Message, Context, Functions

In [21]:
from pathlib import Path



from autogen import LLMConfig



import copy
import os

from pydantic import BaseModel, Field

from autogen import ConversableAgent, UpdateSystemMessage, UserProxyAgent
from autogen.agentchat import a_initiate_group_chat,initiate_group_chat,register_function
from autogen.agentchat.group import (
    AgentTarget,
    ContextVariables,
    ReplyResult,
    OnCondition,
    StringLLMCondition,
    TerminateTarget,
)
from autogen.agentchat.group.patterns import (
    DefaultPattern,
)


In [22]:
# import autogen
# autogen.__file__

In [23]:
coder_message = """
You are the coder in the team. You write code.

You must obey the following constraints:

{code_constraints}

""".strip()



class CoderResponse(BaseModel):
    code: str = Field(..., description="Python code ready to be executed")
    code_explanation: str = Field(..., description="explanation of what the code does")

    def format(self) -> str:
        return "\n".join([
            "**Code:**",
            "",
            self.code,
            "",
            "**Code explanation:**",
            "",
            self.code_explanation,
        ])



llm_config = LLMConfig(
    api_type="openai",
    model="gpt-4o-mini",
    api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.2, 
    # tool_choice = "auto"
)

In [24]:
llm_config.config_list[0].api_type

'openai'

In [25]:
coder_llm_config = copy.deepcopy(llm_config)
coder_llm_config.response_format=CoderResponse


coder = ConversableAgent(
    name="coder",
    system_message=coder_message,
    llm_config=coder_llm_config,
    update_agent_state_before_reply=[
        UpdateSystemMessage(coder_message),
    ],
)


executor_message = """
You are the code executor in the team. You execute code.

""".strip()


executor_llm_config = copy.deepcopy(llm_config)
executor_llm_config.config_list[0]["tool_choice"] = "required"


executor = ConversableAgent(
    name="executor",
    system_message=executor_message,
    llm_config=executor_llm_config,
    update_agent_state_before_reply=[
        UpdateSystemMessage(executor_message),
    ],
)

workflow_context = ContextVariables(
    data={
        "code_constraints": "You must define a function to solve the task, but any function you define must not have any docstrings.",
    }
)




task = """
What sort of code should we write?
"""





user_proxy = UserProxyAgent(
    name="Admin",
    human_input_mode="ALWAYS",
    code_execution_config=False,
)


user_proxy.handoffs.set_after_work(AgentTarget(coder))
coder.handoffs.set_after_work(AgentTarget(executor))
# executor.handoffs.set_after_work(TerminateTarget())


initial_agent = user_proxy

In [28]:
# dir(user_proxy)

In [35]:
agents = [
    coder,
    user_proxy,
    executor
]

for agent in agents:
    agent.reset()




# Create the pattern
agent_pattern = DefaultPattern(
    agents=agents,
    initial_agent=initial_agent,
    context_variables=workflow_context,
)



import os
import shutil

def delete_cache_folder():
    cache_path = os.path.join(os.getcwd(), ".cache")
    if os.path.isdir(cache_path):
        shutil.rmtree(cache_path)
        print(".cache folder deleted.")
    else:
        print("No .cache folder found in current directory.")

delete_cache_folder()


from typing import Annotated

from IPython import get_ipython


# another way of registering functions is to use the register_function
def exec_python(cell: Annotated[str, "Valid Python cell to execute."],context_variables:ContextVariables) -> str:
    ipython = get_ipython()
    result = ipython.run_cell(cell)
    log = str(result.result)
    if result.error_before_exec is not None:
        log += f"\n{result.error_before_exec}"
    if result.error_in_exec is not None:
        log += f"\n{result.error_in_exec}"



    return ReplyResult(target=AgentTarget(user_proxy), ## transfer to planner
                        message=f"The execution result is: {log}",
                        context_variables=context_variables)


    
register_function(
    exec_python,
    caller=executor,
    executor=executor,
    name="python_func",
    description="run cell in ipython and return the execution result.",
)


chat_result, context_variables, last_agent = initiate_group_chat(
        pattern=agent_pattern,
        messages=task,
        max_rounds=20,
    )

No .cache folder found in current directory.
_User (to chat_manager):


What sort of code should we write?


--------------------------------------------------------------------------------

Next speaker: Admin



/Users/boris/CMBAgents/MCPAgents/ag2/autogen/agentchat/conversable_agent.py:3434: UserWarning: Function 'python_func' is being overridden.
  warnings.warn(f"Function '{tool_sig['function']['name']}' is being overridden.", UserWarning)
/Users/boris/CMBAgents/MCPAgents/ag2/autogen/agentchat/conversable_agent.py:3332: UserWarning: Function 'python_func' is being overridden.
  warnings.warn(f"Function '{name}' is being overridden.", UserWarning)


Replying as Admin. Provide feedback to chat_manager. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  draw 2random numbers


Admin (to chat_manager):

draw 2random numbers

--------------------------------------------------------------------------------

Next speaker: coder


>>>>>>>> USING AUTO REPLY...
coder (to chat_manager):

{"code":"import random\n\ndef draw_random_numbers():\n    return random.sample(range(1, 101), 2)","code_explanation":"This code defines a function that draws two random numbers from the range of 1 to 100. It uses the random.sample method to ensure that the two numbers are unique."}

--------------------------------------------------------------------------------

Next speaker: executor


>>>>>>>> USING AUTO REPLY...
executor (to chat_manager):

***** Suggested tool call (call_BN522PfGzev1onTZr1gqsc9Z): python_func *****
Arguments: 
{"cell":"import random\n\ndef draw_random_numbers():\n    return random.sample(range(1, 101), 2)\n\ndraw_random_numbers()"}
****************************************************************************

-----------------------------------------------------

[92, 91]


>>>>>>>> EXECUTED FUNCTION python_func...
Call ID: call_BN522PfGzev1onTZr1gqsc9Z
Input arguments: {'cell': 'import random\n\ndef draw_random_numbers():\n    return random.sample(range(1, 101), 2)\n\ndraw_random_numbers()'}
Output:
The execution result is: [92, 91]
_Group_Tool_Executor (to chat_manager):

***** Response from calling tool (call_BN522PfGzev1onTZr1gqsc9Z) *****
The execution result is: [92, 91]
**********************************************************************

--------------------------------------------------------------------------------

Next speaker: Admin



Replying as Admin. Provide feedback to chat_manager. Press enter to skip and use auto-reply, or type 'exit' to end the conversation:  exit



>>>>>>>> TERMINATING RUN (de01fa9b-89ec-4514-96d7-768faf3f9a06): User requested to end the conversation

>>>>>>>> TERMINATING RUN (26d77a94-3ae3-44c7-8390-676a0241f3ee): No reply generated


In [30]:
# dir(last_agent)

In [31]:
agents

In [32]:
import pprint
for agent in agents:
    print(agent.name)
    pprint.pprint(agent.get_actual_usage())

coder
{'gpt-4o-mini-2024-07-18': {'completion_tokens': 63,
                            'cost': 5.9399999999999994e-05,
                            'prompt_tokens': 144,
                            'total_tokens': 207},
 'total_cost': 5.9399999999999994e-05}
Admin
None
executor
{'gpt-4o-mini-2024-07-18': {'completion_tokens': 51,
                            'cost': 5.46e-05,
                            'prompt_tokens': 160,
                            'total_tokens': 211},
 'total_cost': 5.46e-05}


In [33]:
chat_result.chat_history

[{'content': '\nWhat sort of code should we write?\n', 'role': 'assistant'},
 {'content': 'drwaw two random numbers', 'name': 'Admin', 'role': 'user'},
 {'content': '{"code":"import random\\n\\ndef draw_two_random_numbers():\\n    return random.randint(1, 100), random.randint(1, 100)","code_explanation":"This code imports the random module and defines a function that generates and returns two random integers between 1 and 100."}',
  'name': 'coder',
  'role': 'user'},
 {'content': 'None',
  'tool_calls': [{'id': 'call_jmag0fOWvFKWyYzTdL6DHSwu',
    'function': {'arguments': '{"cell":"import random\\n\\ndef draw_two_random_numbers():\\n    return random.randint(1, 100), random.randint(1, 100)\\n\\ndraw_two_random_numbers()"}',
     'name': 'python_func'},
    'type': 'function'}],
  'name': 'executor',
  'role': 'assistant'},
 {'content': 'The execution result is: (77, 67)',
  'tool_responses': [{'tool_call_id': 'call_jmag0fOWvFKWyYzTdL6DHSwu',
    'role': 'tool',
    'content': 'The ex

# N-shot prompting

In [34]:
result = user_proxy.initiate_chat(
    coder,
    message="Draw two agents chatting with each other with an example dialog. Don't add plt.show().",
    max_turns=1,
)

Admin (to coder):

Draw two agents chatting with each other with an example dialog. Don't add plt.show().

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
coder (to Admin):

{"code":"import matplotlib.pyplot as plt\nimport numpy as np\n\ndef draw_agents_chat():\n    fig, ax = plt.subplots()\n    ax.set_xlim(0, 10)\n    ax.set_ylim(0, 10)\n\n    # Draw Agent 1\n    agent1 = plt.Circle((2, 5), 0.5, color='blue', label='Agent 1')\n    ax.add_artist(agent1)\n    ax.text(2, 5, 'Agent 1', fontsize=12, ha='center', va='center')\n\n    # Draw Agent 2\n    agent2 = plt.Circle((8, 5), 0.5, color='green', label='Agent 2')\n    ax.add_artist(agent2)\n    ax.text(8, 5, 'Agent 2', fontsize=12, ha='center', va='center')\n\n    # Draw speech bubbles\n    ax.annotate('Hello! How are you?', xy=(2.5, 5.5), xytext=(3, 7),\n                arrowprops=dict(facecolor='black', shrink=0.05))\n    ax.annotate('I am good, thanks! And you?', xy=(7.5, 

In [35]:
import json
from IPython.display import display, Markdown, Code

def display_and_run_code(result, exec_scope=None):
    """
    Extracts, displays, and executes code and its explanation from a coder message.
    Parameters:
        result: your result object containing chat_history
        exec_scope: dict, optional scope to execute code in (defaults to globals())
    """
    code_json_str = None
    for msg in result.chat_history[::-1]:
        if msg['name'] == 'coder':
            code_json_str = msg['content']
            break

    if code_json_str is None:
        print("No coder message found.")
        return

    try:
        code_data = json.loads(code_json_str)
        code = code_data.get('code', '')
        explanation = code_data.get('code_explanation', '')
    except Exception as e:
        print(f"Failed to parse code JSON: {e}")
        return

    display(Markdown("**Generated code:**"))
    display(Code(code, language='python'))
    if explanation:
        display(Markdown("**Explanation:**"))
        display(Markdown(explanation))
    
    # Execute the code in the provided scope (or globals by default)
    if code.strip():
        if exec_scope is None:
            exec_scope = globals()
        try:
            exec(code, exec_scope)
            display(Markdown("✅ **Code executed successfully.**"))
        except Exception as e:
            display(Markdown(f"❌ **Error during code execution:** `{e}`"))

# Usage:
# display_and_run_code(result)


In [36]:
display_and_run_code(result)


**Generated code:**

import matplotlib.pyplot as plt
import numpy as np

def draw_agents_chat():
    fig, ax = plt.subplots()
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 10)

    # Draw Agent 1
    agent1 = plt.Circle((2, 5), 0.5, color='blue', label='Agent 1')
    ax.add_artist(agent1)
    ax.text(2, 5, 'Agent 1', fontsize=12, ha='center', va='center')

    # Draw Agent 2
    agent2 = plt.Circle((8, 5), 0.5, color='green', label='Agent 2')
    ax.add_artist(agent2)
    ax.text(8, 5, 'Agent 2', fontsize=12, ha='center', va='center')

    # Draw speech bubbles
    ax.annotate('Hello! How are you?', xy=(2.5, 5.5), xytext=(3, 7),
                arrowprops=dict(facecolor='black', shrink=0.05))
    ax.annotate('I am good, thanks! And you?', xy=(7.5, 5.5), xytext=(6, 7),
                arrowprops=dict(facecolor='black', shrink=0.05))

    ax.set_title('Agents Chatting')
    ax.axis('off')
    plt.show()

**Explanation:**

The code defines a function that uses Matplotlib to draw two agents represented as circles on a plot. Agent 1 is colored blue and located on the left, while Agent 2 is green and located on the right. Each agent has a speech bubble indicating a dialog between them. The speech bubbles show a simple conversation: 'Hello! How are you?' from Agent 1 and 'I am good, thanks! And you?' from Agent 2. The plot is set up with limits and titles, and the axes are turned off for a cleaner look.

✅ **Code executed successfully.**

## 0-shot

In [18]:
result = user_proxy.initiate_chat(
    coder,
    message="""
You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Response format**  
Return only the complete, runnable code for the function – no commentary.
""",
    max_turns=1,
)

display_and_run_code(result)

Admin (to coder):


You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Response format**  
Return only the complete, runnable code for the function – no commentary.


--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
coder (to Admin):

{"code":"def is_palindrome(s: str) -> bool:\n    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())\n    return filtered_chars == filtered_chars[::-1]","code_explanation":"The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting it to lowercase, and comparing it to its reverse."}

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (21cdbb0c-5466-41f6-acdd-67739848349f): Maximum turns (1) reached


**Generated code:**

def is_palindrome(s: str) -> bool:
    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())
    return filtered_chars == filtered_chars[::-1]

**Explanation:**

The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting it to lowercase, and comparing it to its reverse.

✅ **Code executed successfully.**

## 1-shot

In [19]:
result = user_proxy.initiate_chat(
    coder,
    message="""
You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Example**  
Input:  "A man, a plan, a canal: Panama"  
Output: True

**Response format**  
Return only the complete, runnable code for the function – no commentary.
""",
    max_turns=1,
)

display_and_run_code(result)

Admin (to coder):


You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Example**  
Input:  "A man, a plan, a canal: Panama"  
Output: True

**Response format**  
Return only the complete, runnable code for the function – no commentary.


--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
coder (to Admin):

{"code":"def is_palindrome(s: str) -> bool:\n    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())\n    return filtered_chars == filtered_chars[::-1]","code_explanation":"The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting it to lowercase, and comparing it to its reverse."}

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING 

**Generated code:**

def is_palindrome(s: str) -> bool:
    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())
    return filtered_chars == filtered_chars[::-1]

**Explanation:**

The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting it to lowercase, and comparing it to its reverse.

✅ **Code executed successfully.**

In [20]:
result = user_proxy.initiate_chat(
    coder,
    message="""
You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Examples**  
1️⃣ Input: "A man, a plan, a canal: Panama"  
   Output: True

2️⃣ Input: "race a car"  
   Output: False

**Response format**  
Return only the complete, runnable code for the function – no commentary.
""",
    max_turns=1,
)

display_and_run_code(result)

Admin (to coder):


You are an expert Python developer.

**Task**  
Write a Python function `is_palindrome(s: str) -> bool` that returns `True` when `s` is a palindrome, **ignoring case and any non-alphanumeric characters**, and `False` otherwise.

**Examples**  
1️⃣ Input: "A man, a plan, a canal: Panama"  
   Output: True

2️⃣ Input: "race a car"  
   Output: False

**Response format**  
Return only the complete, runnable code for the function – no commentary.


--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
coder (to Admin):

{"code":"def is_palindrome(s: str) -> bool:\n    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())\n    return filtered_chars == filtered_chars[::-1]","code_explanation":"The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting the remaining characters to lowercase, and comparing the filtered string to its reverse."}

-------------

**Generated code:**

def is_palindrome(s: str) -> bool:
    filtered_chars = ''.join(c.lower() for c in s if c.isalnum())
    return filtered_chars == filtered_chars[::-1]

**Explanation:**

The function checks if the input string is a palindrome by filtering out non-alphanumeric characters, converting the remaining characters to lowercase, and comparing the filtered string to its reverse.

✅ **Code executed successfully.**

## Other ways to get responses

In [28]:
# Run the chat
chat_result = user_proxy.initiate_chat(
    recipient=coder,
    message="Hello Bob, tell me a joke!",
    max_turns=1
)

Admin (to coder):

Hello Bob, tell me a joke!

--------------------------------------------------------------------------------

>>>>>>>> USING AUTO REPLY...
coder (to Admin):

{"code":"def tell_joke():\n    return \"Why don't scientists trust atoms? Because they make up everything!\"","code_explanation":"This function, when called, returns a joke about atoms, specifically that scientists don't trust them because they 'make up' everything."}

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (b5382279-7343-4ee4-a145-ffdd8dc48ca4): Maximum turns (1) reached


In [16]:
prompt = "write code about love"
result = coder.generate_reply(messages=[{"role":"user", "content":prompt}])


>>>>>>>> USING AUTO REPLY...


In [27]:
coder.reset()

In [29]:
coder.chat_messages

defaultdict(list,
            {<autogen.agentchat.user_proxy_agent.UserProxyAgent at 0x137cd7770>: [{'content': 'Hello Bob, tell me a joke!',
               'role': 'user',
               'name': 'Admin'},
              {'content': '{"code":"def tell_joke():\\n    return \\"Why don\'t scientists trust atoms? Because they make up everything!\\"","code_explanation":"This function, when called, returns a joke about atoms, specifically that scientists don\'t trust them because they \'make up\' everything."}',
               'role': 'assistant',
               'name': 'coder'}]})